
# Create a Patient-Level CSV from Standard Synthea CSV Files

This notebook converts the **standard Synthea CSV export** into a single patient-level tabular file that can be used directly with GrEAt or other tabular anomaly detection models.

It expects files such as:

- `patients.csv`
- `encounters.csv`
- `conditions.csv`
- `medications.csv`
- `procedures.csv`
- `observations.csv`
- optional: `allergies.csv`, `immunizations.csv`, `careplans.csv`, `devices.csv`

The output is one CSV file where each row represents one patient and columns are aggregated demographic, encounter, condition, medication, procedure, observation, and optional clinical-event features.


In [1]:

import os
import warnings
from functools import reduce

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)



## 1. Configuration

Set `DATA_DIR` to the folder containing your Synthea CSV files.

The default label is `high_utilization`, which marks the top 10% of patients by number of encounters as anomalies. This is usually more practical than `death`, because death may be rare or absent depending on the Synthea cohort.


In [2]:

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
DATA_DIR = "./"  # <-- change this to your folder path
OUTPUT_CSV = "synthea_patient_level.csv"

# Label options:
#   "high_utilization" = top quantile of encounter count is anomaly
#   "death"            = patients with non-empty DEATHDATE are anomaly
LABEL_MODE = "high_utilization"
HIGH_UTILIZATION_QUANTILE = 0.90

# Top-k indicators control dimensionality.
TOP_K_CONDITIONS = 50
TOP_K_MEDICATIONS = 50
TOP_K_PROCEDURES = 50
TOP_K_OBSERVATIONS = 30
TOP_K_ENCOUNTER_CODES = 20

# Optional event files. The notebook will use them if present.
USE_OPTIONAL_FILES = True

# Keep patient_id in the final CSV for traceability. Your GrEAt loader can drop it as an ID column.
KEEP_PATIENT_ID = True

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_CSV:", OUTPUT_CSV)
print("LABEL_MODE:", LABEL_MODE)


DATA_DIR: ./
OUTPUT_CSV: synthea_patient_level.csv
LABEL_MODE: high_utilization


## 2. Helper functions

In [3]:

def read_csv_if_exists(data_dir, filename, required=False):
    path = os.path.join(data_dir, filename)
    if os.path.exists(path):
        df = pd.read_csv(path, low_memory=False)
        print(f"Loaded {filename}: {df.shape}")
        return df
    if required:
        raise FileNotFoundError(f"Required file not found: {path}")
    print(f"Optional file not found, skipping: {filename}")
    return None


def clean_colname(x):
    """Make column names safe for ML pipelines and LaTeX tables."""
    x = str(x)
    x = x.strip().lower()
    x = x.replace(" ", "_").replace("-", "_").replace("/", "_")
    x = "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in x)
    while "__" in x:
        x = x.replace("__", "_")
    return x.strip("_")


def add_prefix_except_id(df, prefix, id_col="patient_id"):
    out = df.copy()
    out.columns = [id_col if c == id_col else f"{prefix}{clean_colname(c)}" for c in out.columns]
    return out


def aggregate_counts(df, patient_col="PATIENT", id_col=None, prefix=""):
    """Basic event count aggregation by patient."""
    if df is None or patient_col not in df.columns:
        return None
    tmp = df.copy()
    if id_col is not None and id_col in tmp.columns:
        count_col = id_col
    else:
        tmp["_row"] = 1
        count_col = "_row"
    agg = tmp.groupby(patient_col).agg(
        total=(count_col, "count")
    ).reset_index().rename(columns={patient_col: "patient_id"})
    return add_prefix_except_id(agg, prefix)


def aggregate_unique(df, cols, patient_col="PATIENT", prefix=""):
    """Number of unique values for selected columns by patient."""
    if df is None or patient_col not in df.columns:
        return None
    present_cols = [c for c in cols if c in df.columns]
    if not present_cols:
        return None
    agg_spec = {f"n_unique_{clean_colname(c)}": (c, "nunique") for c in present_cols}
    out = df.groupby(patient_col).agg(**agg_spec).reset_index().rename(columns={patient_col: "patient_id"})
    return add_prefix_except_id(out, prefix)


def topk_indicators(df, code_col, patient_col="PATIENT", top_k=50, prefix=""):
    """Create binary patient-level indicators for top-k codes/descriptions."""
    if df is None or patient_col not in df.columns or code_col not in df.columns:
        return None
    tmp = df[[patient_col, code_col]].dropna().copy()
    if tmp.empty:
        return None
    top_values = tmp[code_col].astype(str).value_counts().head(top_k).index
    tmp = tmp[tmp[code_col].astype(str).isin(top_values)].copy()
    tmp["value"] = 1
    piv = tmp.pivot_table(index=patient_col, columns=code_col, values="value", aggfunc="max", fill_value=0)
    piv.columns = [f"{prefix}{clean_colname(c)}" for c in piv.columns]
    piv = piv.reset_index().rename(columns={patient_col: "patient_id"})
    return piv


def numeric_summary(df, patient_col, numeric_cols, prefix):
    """Mean/std/min/max/sum summaries for numeric columns."""
    if df is None or patient_col not in df.columns:
        return None
    present = [c for c in numeric_cols if c in df.columns]
    if not present:
        return None
    tmp = df[[patient_col] + present].copy()
    for c in present:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce")
    agg_spec = {}
    for c in present:
        safe = clean_colname(c)
        agg_spec[f"{prefix}{safe}_mean"] = (c, "mean")
        agg_spec[f"{prefix}{safe}_std"] = (c, "std")
        agg_spec[f"{prefix}{safe}_min"] = (c, "min")
        agg_spec[f"{prefix}{safe}_max"] = (c, "max")
        agg_spec[f"{prefix}{safe}_sum"] = (c, "sum")
    out = tmp.groupby(patient_col).agg(**agg_spec).reset_index().rename(columns={patient_col: "patient_id"})
    return out


def observation_features(observations, top_k=30):
    """Aggregate Synthea observations.csv to patient-level numeric features."""
    if observations is None or "PATIENT" not in observations.columns:
        return None
    obs = observations.copy()
    if "VALUE" not in obs.columns:
        return None
    obs["VALUE_NUM"] = pd.to_numeric(obs["VALUE"], errors="coerce")
    obs_num = obs.dropna(subset=["VALUE_NUM"]).copy()
    if obs_num.empty:
        return None

    # General numeric observation summaries.
    general = obs_num.groupby("PATIENT").agg(
        obs_n_numeric=("VALUE_NUM", "count"),
        obs_n_unique_codes=("CODE", "nunique") if "CODE" in obs_num.columns else ("VALUE_NUM", "count"),
        obs_value_mean=("VALUE_NUM", "mean"),
        obs_value_std=("VALUE_NUM", "std"),
        obs_value_min=("VALUE_NUM", "min"),
        obs_value_max=("VALUE_NUM", "max"),
    ).reset_index().rename(columns={"PATIENT": "patient_id"})

    # Measurement-specific means for top-k observation codes.
    if "CODE" in obs_num.columns:
        top_codes = obs_num["CODE"].astype(str).value_counts().head(top_k).index
        top = obs_num[obs_num["CODE"].astype(str).isin(top_codes)].copy()
        piv_mean = top.pivot_table(index="PATIENT", columns="CODE", values="VALUE_NUM", aggfunc="mean")
        piv_mean.columns = [f"obs_mean_{clean_colname(c)}" for c in piv_mean.columns]
        piv_mean = piv_mean.reset_index().rename(columns={"PATIENT": "patient_id"})
        return general.merge(piv_mean, on="patient_id", how="left")

    return general


def parse_datetime_naive(x):
    """Parse dates/timestamps and return timezone-naive pandas datetimes.

    Synthea CSV files sometimes mix date-only strings such as 1980-01-01
    with timezone-aware timestamps such as 2020-01-01T12:00:00Z.
    Using utc=True makes them comparable, and tz_localize(None) removes
    the timezone so pandas can safely subtract them.
    """
    dt = pd.to_datetime(x, errors="coerce", utc=True)
    if isinstance(dt, pd.Series):
        return dt.dt.tz_localize(None)
    if isinstance(dt, pd.DatetimeIndex):
        return dt.tz_localize(None)
    if pd.isna(dt):
        return pd.NaT
    return dt.tz_localize(None)


def date_diff_days(start, stop):
    s = parse_datetime_naive(start)
    e = parse_datetime_naive(stop)
    return (e - s).dt.total_seconds() / (24 * 3600)


## 3. Load Synthea CSV files

In [4]:

patients = read_csv_if_exists(DATA_DIR, "patients.csv", required=True)
encounters = read_csv_if_exists(DATA_DIR, "encounters.csv", required=True)
conditions = read_csv_if_exists(DATA_DIR, "conditions.csv", required=False)
medications = read_csv_if_exists(DATA_DIR, "medications.csv", required=False)
procedures = read_csv_if_exists(DATA_DIR, "procedures.csv", required=False)
observations = read_csv_if_exists(DATA_DIR, "observations.csv", required=False)

if USE_OPTIONAL_FILES:
    allergies = read_csv_if_exists(DATA_DIR, "allergies.csv", required=False)
    immunizations = read_csv_if_exists(DATA_DIR, "immunizations.csv", required=False)
    careplans = read_csv_if_exists(DATA_DIR, "careplans.csv", required=False)
    devices = read_csv_if_exists(DATA_DIR, "devices.csv", required=False)
else:
    allergies = immunizations = careplans = devices = None


Loaded patients.csv: (1171, 25)
Loaded encounters.csv: (53346, 15)
Loaded conditions.csv: (8376, 6)
Loaded medications.csv: (42989, 13)
Loaded procedures.csv: (34981, 8)
Loaded observations.csv: (299697, 8)
Loaded allergies.csv: (597, 6)
Loaded immunizations.csv: (15478, 6)
Loaded careplans.csv: (3483, 9)
Loaded devices.csv: (78, 7)


## 4. Build patient-level base table

In [5]:

# ---------------------------------------------------------------------
# Patient-level base table
# ---------------------------------------------------------------------
if "Id" not in patients.columns:
    raise ValueError("patients.csv must contain an 'Id' column.")

base = patients.copy()
base = base.rename(columns={"Id": "patient_id"})

# Age from BIRTHDATE.
if "BIRTHDATE" in base.columns:
    birth = parse_datetime_naive(base["BIRTHDATE"])
    # Use latest available date in encounters as reference, else today.
    if encounters is not None and "START" in encounters.columns:
        ref_date = parse_datetime_naive(encounters["START"]).max()
        if pd.isna(ref_date):
            ref_date = pd.Timestamp.today().tz_localize(None)
    else:
        ref_date = pd.Timestamp.today().tz_localize(None)
    base["age"] = ((ref_date - birth).dt.days / 365.25).clip(lower=0)

# Death indicator. This can be used as label if LABEL_MODE='death'.
if "DEATHDATE" in base.columns:
    base["death_indicator"] = base["DEATHDATE"].notna().astype(int)
else:
    base["death_indicator"] = 0

# Keep useful demographic/socioeconomic columns when available.
candidate_cols = [
    "patient_id", "age", "death_indicator",
    "GENDER", "RACE", "ETHNICITY", "MARITAL", "STATE", "COUNTY",
    "HEALTHCARE_EXPENSES", "HEALTHCARE_COVERAGE", "INCOME",
    "LAT", "LON"
]
base_cols = [c for c in candidate_cols if c in base.columns]
base = base[base_cols].copy()

print("Base table:", base.shape)
base.head()


Base table: (1171, 13)


,patient_id,age,death_indicator,GENDER,RACE,ETHNICITY,MARITAL,STATE,COUNTY,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,LAT,LON
0,1d604da9-9a81-4ba9-80c2-de3375d59b40,30.926762,0,M,white,hispanic,M,Massachusetts,Hampden County,271227.08,1334.88,42.228354,-72.562951
1,034e9e3b-2def-4559-bb2a-7850888ae060,36.454483,0,M,white,nonhispanic,M,Massachusetts,Middlesex County,793946.01,3204.49,42.360697,-71.126531
2,10339b10-3cd1-4ac3-ac13-ec26728cb592,27.904175,0,M,white,nonhispanic,M,Massachusetts,Hampden County,574111.90,2606.40,42.181642,-72.608842
3,8d4c4326-e9de-4f45-9a4c-f8c36bff89ae,41.921971,0,F,white,nonhispanic,M,Massachusetts,Middlesex County,935630.30,8756.19,42.636143,-71.343255
4,f5dcd418-09fe-4a2f-baa0-3da800bd8c3a,23.526352,0,M,white,nonhispanic,NaN,Massachusetts,Suffolk County,598763.07,3772.20,42.352434,-71.028610


## 5. Aggregate encounters

In [6]:

enc = encounters.copy()

# Encounter durations if START/STOP exist.
if "START" in enc.columns and "STOP" in enc.columns:
    enc["encounter_duration_days"] = date_diff_days(enc["START"], enc["STOP"])

enc_count = aggregate_counts(enc, patient_col="PATIENT", id_col="Id" if "Id" in enc.columns else None, prefix="enc_")
enc_unique = aggregate_unique(enc, ["ENCOUNTERCLASS", "CODE", "REASONCODE", "PROVIDER", "PAYER"], patient_col="PATIENT", prefix="enc_")
enc_numeric = numeric_summary(
    enc,
    patient_col="PATIENT",
    numeric_cols=["BASE_ENCOUNTER_COST", "TOTAL_CLAIM_COST", "PAYER_COVERAGE", "encounter_duration_days"],
    prefix="enc_"
)
enc_class_ind = topk_indicators(enc, "ENCOUNTERCLASS", patient_col="PATIENT", top_k=20, prefix="enc_class_")
enc_code_ind = topk_indicators(enc, "CODE", patient_col="PATIENT", top_k=TOP_K_ENCOUNTER_CODES, prefix="enc_code_")

enc_tables = [t for t in [enc_count, enc_unique, enc_numeric, enc_class_ind, enc_code_ind] if t is not None]
enc_features = reduce(lambda l, r: l.merge(r, on="patient_id", how="outer"), enc_tables)

print("Encounter features:", enc_features.shape)
enc_features.head()


Encounter features: (1171, 53)


,patient_id,enc_total,enc_n_unique_encounterclass,enc_n_unique_code,enc_n_unique_reasoncode,enc_n_unique_provider,enc_n_unique_payer,enc_base_encounter_cost_mean,enc_base_encounter_cost_std,enc_base_encounter_cost_min,enc_base_encounter_cost_max,enc_base_encounter_cost_sum,enc_total_claim_cost_mean,enc_total_claim_cost_std,enc_total_claim_cost_min,enc_total_claim_cost_max,enc_total_claim_cost_sum,enc_payer_coverage_mean,enc_payer_coverage_std,enc_payer_coverage_min,enc_payer_coverage_max,enc_payer_coverage_sum,enc_encounter_duration_days_mean,enc_encounter_duration_days_std,enc_encounter_duration_days_min,enc_encounter_duration_days_max,enc_encounter_duration_days_sum,enc_class_ambulatory,enc_class_emergency,enc_class_inpatient,enc_class_outpatient,enc_class_urgentcare,enc_class_wellness,enc_code_50849002,enc_code_56876005,enc_code_162673000,enc_code_169762003,enc_code_183452005,enc_code_183460006,enc_code_185345009,enc_code_185347001,enc_code_185349003,enc_code_308335008,enc_code_371883000,enc_code_390906007,enc_code_394701000,enc_code_410620009,enc_code_424441002,enc_code_424619006,enc_code_439740005,enc_code_448337001,enc_code_698314001,enc_code_702927004
0,00185faa-2760-4218-9bf5-db301acf8274,22,3,3,5,2,1,105.673636,26.333470,77.49,129.16,2324.82,105.673636,26.333470,77.49,129.16,2324.82,64.764545,61.903861,2.49,129.16,1424.82,0.464962,0.502860,0.010417,1.010417,10.229167,1,0,1,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0
1,0042862c-9889-4a2e-b782-fac1e540ecb4,21,4,3,3,3,2,129.160000,0.000000,129.16,129.16,2712.36,129.160000,0.000000,129.16,129.16,2712.36,40.632381,56.164957,0.00,129.16,853.28,0.015046,0.005527,0.010417,0.024306,0.315972,1,0,0,1,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1
2,0047123f-12e7-486c-82df-53b3a450e365,21,6,6,3,3,1,126.699524,11.275318,77.49,129.16,2660.69,126.699524,11.275318,77.49,129.16,2660.69,85.390952,48.727192,0.00,129.16,1793.21,0.207407,0.878594,0.010417,4.041667,4.355556,1,1,1,1,1,1,1,0,1,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1
3,010d4a3a-2316-45ed-ae15-16f01c611674,13,2,4,1,2,1,129.160000,0.000000,129.16,129.16,1679.08,129.160000,0.000000,129.16,129.16,1679.08,113.006154,30.697031,59.16,129.16,1469.08,0.012019,0.003912,0.010417,0.020833,0.156250,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0
4,01207ecd-9dff-4754-8887-4652eda231e2,6,1,1,0,1,1,129.160000,0.000000,129.16,129.16,774.96,129.160000,0.000000,129.16,129.16,774.96,129.160000,0.000000,129.16,129.16,774.96,0.017361,0.005379,0.010417,0.020833,0.104167,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0


## 6. Aggregate conditions, medications, procedures, observations, and optional clinical-event tables

In [7]:

# Conditions
cond_count = aggregate_counts(conditions, patient_col="PATIENT", prefix="cond_")
cond_unique = aggregate_unique(conditions, ["CODE", "DESCRIPTION"], patient_col="PATIENT", prefix="cond_")
cond_ind = topk_indicators(conditions, "CODE", patient_col="PATIENT", top_k=TOP_K_CONDITIONS, prefix="cond_code_")
cond_tables = [t for t in [cond_count, cond_unique, cond_ind] if t is not None]
cond_features = reduce(lambda l, r: l.merge(r, on="patient_id", how="outer"), cond_tables) if cond_tables else None
print("Condition features:", None if cond_features is None else cond_features.shape)

# Medications
med_count = aggregate_counts(medications, patient_col="PATIENT", prefix="med_")
med_unique = aggregate_unique(medications, ["CODE", "DESCRIPTION", "REASONCODE", "PAYER"], patient_col="PATIENT", prefix="med_")
med_numeric = numeric_summary(medications, "PATIENT", ["BASE_COST", "PAYER_COVERAGE", "DISPENSES", "TOTALCOST"], "med_")
med_ind = topk_indicators(medications, "CODE", patient_col="PATIENT", top_k=TOP_K_MEDICATIONS, prefix="med_code_")
med_tables = [t for t in [med_count, med_unique, med_numeric, med_ind] if t is not None]
med_features = reduce(lambda l, r: l.merge(r, on="patient_id", how="outer"), med_tables) if med_tables else None
print("Medication features:", None if med_features is None else med_features.shape)

# Procedures
proc_count = aggregate_counts(procedures, patient_col="PATIENT", prefix="proc_")
proc_unique = aggregate_unique(procedures, ["CODE", "DESCRIPTION", "REASONCODE"], patient_col="PATIENT", prefix="proc_")
proc_numeric = numeric_summary(procedures, "PATIENT", ["BASE_COST"], "proc_")
proc_ind = topk_indicators(procedures, "CODE", patient_col="PATIENT", top_k=TOP_K_PROCEDURES, prefix="proc_code_")
proc_tables = [t for t in [proc_count, proc_unique, proc_numeric, proc_ind] if t is not None]
proc_features = reduce(lambda l, r: l.merge(r, on="patient_id", how="outer"), proc_tables) if proc_tables else None
print("Procedure features:", None if proc_features is None else proc_features.shape)

# Observations
obs_features = observation_features(observations, top_k=TOP_K_OBSERVATIONS)
print("Observation features:", None if obs_features is None else obs_features.shape)

# Optional tables
optional_feature_tables = []
if USE_OPTIONAL_FILES:
    for name, df_opt, code_col, top_k in [
        ("allergy", allergies, "CODE", 30),
        ("imm", immunizations, "CODE", 30),
        ("care", careplans, "CODE", 30),
        ("device", devices, "CODE", 30),
    ]:
        c = aggregate_counts(df_opt, patient_col="PATIENT", prefix=f"{name}_")
        u = aggregate_unique(df_opt, [code_col, "DESCRIPTION"], patient_col="PATIENT", prefix=f"{name}_")
        ind = topk_indicators(df_opt, code_col, patient_col="PATIENT", top_k=top_k, prefix=f"{name}_code_")
        tables = [t for t in [c, u, ind] if t is not None]
        if tables:
            feat = reduce(lambda l, r: l.merge(r, on="patient_id", how="outer"), tables)
            optional_feature_tables.append(feat)
            print(f"{name} features:", feat.shape)


Condition features: (1152, 54)
Medication features: (1107, 76)
Procedure features: (1165, 60)
Observation features: (1171, 37)
allergy features: (141, 19)
imm features: (1169, 22)
care features: (1054, 34)
device features: (74, 7)


## 7. Merge all patient-level feature tables

In [8]:

feature_tables = [
    base,
    enc_features,
    cond_features,
    med_features,
    proc_features,
    obs_features,
] + optional_feature_tables

feature_tables = [t for t in feature_tables if t is not None]

df = reduce(lambda left, right: left.merge(right, on="patient_id", how="left"), feature_tables)

# Fill missing numeric event features with 0, because missing often means no such event.
# Keep categorical missing values as "Unknown".
for c in df.columns:
    if c == "patient_id":
        continue
    if df[c].dtype == "object":
        df[c] = df[c].fillna("Unknown")
    else:
        df[c] = df[c].fillna(0)

print("Merged patient-level table:", df.shape)
df.head()


Merged patient-level table: (1171, 366)


,patient_id,age,death_indicator,GENDER,RACE,ETHNICITY,MARITAL,STATE,COUNTY,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,LAT,LON,enc_total,enc_n_unique_encounterclass,enc_n_unique_code,enc_n_unique_reasoncode,enc_n_unique_provider,enc_n_unique_payer,enc_base_encounter_cost_mean,enc_base_encounter_cost_std,enc_base_encounter_cost_min,enc_base_encounter_cost_max,enc_base_encounter_cost_sum,enc_total_claim_cost_mean,enc_total_claim_cost_std,enc_total_claim_cost_min,enc_total_claim_cost_max,enc_total_claim_cost_sum,enc_payer_coverage_mean,enc_payer_coverage_std,enc_payer_coverage_min,enc_payer_coverage_max,enc_payer_coverage_sum,enc_encounter_duration_days_mean,enc_encounter_duration_days_std,enc_encounter_duration_days_min,enc_encounter_duration_days_max,enc_encounter_duration_days_sum,enc_class_ambulatory,enc_class_emergency,enc_class_inpatient,enc_class_outpatient,enc_class_urgentcare,enc_class_wellness,enc_code_50849002,enc_code_56876005,enc_code_162673000,enc_code_169762003,enc_code_183452005,enc_code_183460006,enc_code_185345009,enc_code_185347001,enc_code_185349003,enc_code_308335008,enc_code_371883000,enc_code_390906007,enc_code_394701000,enc_code_410620009,enc_code_424441002,enc_code_424619006,enc_code_439740005,enc_code_448337001,enc_code_698314001,enc_code_702927004,cond_total,cond_n_unique_code,cond_n_unique_description,cond_code_10509002,cond_code_15777000,cond_code_19169002,cond_code_36971009,cond_code_39848009,cond_code_40055000,cond_code_43878008,cond_code_44054006,cond_code_44465007,cond_code_49436004,cond_code_53741008,cond_code_55680006,cond_code_55822004,cond_code_59621000,cond_code_62106007,cond_code_64859006,cond_code_65363002,cond_code_65966004,cond_code_68496003,cond_code_70704007,cond_code_72892002,cond_code_74400008,cond_code_75498004,cond_code_79586000,cond_code_80394007,cond_code_82423001,cond_code_88805009,cond_code_127013003,cond_code_128613002,cond_code_156073000,cond_code_162864005,cond_code_195662009,...,obs_mean_2947_0,obs_mean_33914_3,obs_mean_38483_4,obs_mean_39156_5,obs_mean_4548_4,obs_mean_49765_1,obs_mean_59576_9,obs_mean_6298_4,obs_mean_6299_2,obs_mean_6690_2,obs_mean_718_7,obs_mean_72514_3,obs_mean_74006_8,obs_mean_789_8,obs_mean_8302_2,obs_mean_8462_4,obs_mean_8480_6,obs_mean_8867_4,obs_mean_9279_1,obs_mean_daly,obs_mean_qaly,obs_mean_qols,allergy_total,allergy_n_unique_code,allergy_n_unique_description,allergy_code_91930004,allergy_code_91934008,allergy_code_91935009,allergy_code_232347008,allergy_code_232350006,allergy_code_300913006,allergy_code_300916003,allergy_code_417532002,allergy_code_418689008,allergy_code_419263009,allergy_code_419474003,allergy_code_420174000,allergy_code_424213003,allergy_code_425525006,allergy_code_714035009,imm_total,imm_n_unique_code,imm_n_unique_description,imm_code_3,imm_code_8,imm_code_10,imm_code_20,imm_code_21,imm_code_33,imm_code_43,imm_code_49,imm_code_52,imm_code_62,imm_code_83,imm_code_113,imm_code_114,imm_code_115,imm_code_119,imm_code_121,imm_code_133,imm_code_140,care_total,care_n_unique_code,care_n_unique_description,care_code_47387005,care_code_53950000,care_code_91251008,care_code_133901003,care_code_134435003,care_code_170836005,care_code_182964004,care_code_225358003,care_code_384758001,care_code_385691007,care_code_386257007,care_code_386522008,care_code_395082007,care_code_408869004,care_code_412776001,care_code_443402002,care_code_698360004,care_code_699728000,care_code_711282006,care_code_718347000,care_code_734163000,care_code_735984001,care_code_736252007,care_code_736254008,care_code_736285004,care_code_736353004,care_code_736690008,care_code_737471002,care_code_781831000000109,care_code_869761000000107,device_total,device_n_unique_code,device_n_unique_description,device_code_72506001,device_code_705643001,device_code_706004007
0,1d604da9-9a81-4ba9-80c2-de3375d59b40,30.926762,0,M,white,hispanic,M,Massachusetts,Hampden County,271227.08,1334.88,42.228354,-72.562951,6,2,2,2,2,1,129.16,0.0,129.16,129.16,774.96,129.16,0.0,129.16,129.16,774.96,0

## 8. Create anomaly label

In [9]:

# ---------------------------------------------------------------------
# Label creation
# ---------------------------------------------------------------------
if LABEL_MODE == "death":
    df["label"] = df["death_indicator"].astype(int)

elif LABEL_MODE == "high_utilization":
    if "enc_total" not in df.columns:
        raise ValueError("enc_total was not created. Check that encounters.csv loaded correctly.")
    threshold = df["enc_total"].quantile(HIGH_UTILIZATION_QUANTILE)
    df["label"] = (df["enc_total"] >= threshold).astype(int)
    print(f"High-utilization threshold: enc_total >= {threshold:.3f}")

else:
    raise ValueError("LABEL_MODE must be 'death' or 'high_utilization'.")

print("Label distribution:")
print(df["label"].value_counts(dropna=False))
print("Anomaly ratio:", df["label"].mean())


High-utilization threshold: enc_total >= 84.000
Label distribution:
label
0    1046
1     125
Name: count, dtype: int64
Anomaly ratio: 0.1067463706233988


## 9. Clean columns and save final CSV

In [10]:

# Drop raw death indicator if it duplicates the label. Keep it only if using high_utilization and you want it as a feature.
# To avoid leakage when LABEL_MODE='death', remove death_indicator from features.
if LABEL_MODE == "death" and "death_indicator" in df.columns:
    df = df.drop(columns=["death_indicator"])

# Rename all columns to safe names.
rename_map = {c: clean_colname(c) for c in df.columns}
df = df.rename(columns=rename_map)

# Optionally remove patient_id.
if not KEEP_PATIENT_ID and "patient_id" in df.columns:
    df = df.drop(columns=["patient_id"])

# Put label at the end.
cols = [c for c in df.columns if c != "label"] + ["label"]
df = df[cols]

# Remove constant columns except label and patient_id.
constant_cols = []
for c in df.columns:
    if c in ["label", "patient_id"]:
        continue
    if df[c].nunique(dropna=False) <= 1:
        constant_cols.append(c)

print(f"Dropping {len(constant_cols)} constant columns.")
if constant_cols:
    df = df.drop(columns=constant_cols)

# Save.
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")
print("Final shape:", df.shape)
print("Final anomaly ratio:", df["label"].mean())

# Identify numeric/categorical columns for GrEAt.
non_feature_cols = ["label"] + (["patient_id"] if "patient_id" in df.columns else [])
categorical_cols = [c for c in df.columns if c not in non_feature_cols and df[c].dtype == "object"]
numeric_cols = [c for c in df.columns if c not in non_feature_cols + categorical_cols]

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print("Categorical column names:", categorical_cols[:30])

# Show a small preview.
df.head()


Dropping 5 constant columns.
Saved: synthea_patient_level.csv
Final shape: (1171, 362)
Final anomaly ratio: 0.1067463706233988
Numeric columns: 355
Categorical columns: 5
Categorical column names: ['gender', 'race', 'ethnicity', 'marital', 'county']


,patient_id,age,death_indicator,gender,race,ethnicity,marital,county,healthcare_expenses,healthcare_coverage,lat,lon,enc_total,enc_n_unique_encounterclass,enc_n_unique_code,enc_n_unique_reasoncode,enc_n_unique_provider,enc_n_unique_payer,enc_base_encounter_cost_mean,enc_base_encounter_cost_std,enc_base_encounter_cost_min,enc_base_encounter_cost_sum,enc_total_claim_cost_mean,enc_total_claim_cost_std,enc_total_claim_cost_min,enc_total_claim_cost_sum,enc_payer_coverage_mean,enc_payer_coverage_std,enc_payer_coverage_min,enc_payer_coverage_max,enc_payer_coverage_sum,enc_encounter_duration_days_mean,enc_encounter_duration_days_std,enc_encounter_duration_days_max,enc_encounter_duration_days_sum,enc_class_ambulatory,enc_class_emergency,enc_class_inpatient,enc_class_outpatient,enc_class_urgentcare,enc_code_50849002,enc_code_56876005,enc_code_162673000,enc_code_169762003,enc_code_183452005,enc_code_183460006,enc_code_185345009,enc_code_185347001,enc_code_185349003,enc_code_308335008,enc_code_371883000,enc_code_390906007,enc_code_394701000,enc_code_410620009,enc_code_424441002,enc_code_424619006,enc_code_439740005,enc_code_448337001,enc_code_698314001,enc_code_702927004,cond_total,cond_n_unique_code,cond_n_unique_description,cond_code_10509002,cond_code_15777000,cond_code_19169002,cond_code_36971009,cond_code_39848009,cond_code_40055000,cond_code_43878008,cond_code_44054006,cond_code_44465007,cond_code_49436004,cond_code_53741008,cond_code_55680006,cond_code_55822004,cond_code_59621000,cond_code_62106007,cond_code_64859006,cond_code_65363002,cond_code_65966004,cond_code_68496003,cond_code_70704007,cond_code_72892002,cond_code_74400008,cond_code_75498004,cond_code_79586000,cond_code_80394007,cond_code_82423001,cond_code_88805009,cond_code_127013003,cond_code_128613002,cond_code_156073000,cond_code_162864005,cond_code_195662009,cond_code_196416002,cond_code_230690007,cond_code_237602007,cond_code_239873007,cond_code_263102004,...,obs_mean_33914_3,obs_mean_38483_4,obs_mean_39156_5,obs_mean_4548_4,obs_mean_49765_1,obs_mean_59576_9,obs_mean_6298_4,obs_mean_6299_2,obs_mean_6690_2,obs_mean_718_7,obs_mean_72514_3,obs_mean_74006_8,obs_mean_789_8,obs_mean_8302_2,obs_mean_8462_4,obs_mean_8480_6,obs_mean_8867_4,obs_mean_9279_1,obs_mean_daly,obs_mean_qaly,obs_mean_qols,allergy_total,allergy_n_unique_code,allergy_n_unique_description,allergy_code_91930004,allergy_code_91934008,allergy_code_91935009,allergy_code_232347008,allergy_code_232350006,allergy_code_300913006,allergy_code_300916003,allergy_code_417532002,allergy_code_418689008,allergy_code_419263009,allergy_code_419474003,allergy_code_420174000,allergy_code_424213003,allergy_code_425525006,allergy_code_714035009,imm_total,imm_n_unique_code,imm_n_unique_description,imm_code_3,imm_code_8,imm_code_10,imm_code_20,imm_code_21,imm_code_33,imm_code_43,imm_code_49,imm_code_52,imm_code_62,imm_code_83,imm_code_113,imm_code_114,imm_code_115,imm_code_119,imm_code_121,imm_code_133,imm_code_140,care_total,care_n_unique_code,care_n_unique_description,care_code_47387005,care_code_53950000,care_code_91251008,care_code_133901003,care_code_134435003,care_code_170836005,care_code_182964004,care_code_225358003,care_code_384758001,care_code_385691007,care_code_386257007,care_code_386522008,care_code_395082007,care_code_408869004,care_code_412776001,care_code_443402002,care_code_698360004,care_code_699728000,care_code_711282006,care_code_718347000,care_code_734163000,care_code_735984001,care_code_736252007,care_code_736254008,care_code_736285004,care_code_736353004,care_code_736690008,care_code_737471002,care_code_781831000000109,care_code_869761000000107,device_total,device_n_unique_code,device_n_unique_description,device_code_72506001,device_code_705643001,device_code_706004007,label
0,1d604da9-9a81-4ba9-80c2-de3375d59b40,30.926762,0,M,white,hispanic,M,Hampden County,271227.08,1334.88,42.228354,-72.562951,6,2,2,2,2,1,129.16,0.0,129.16,774.96,129.16,0.0,129.16,774.96,0.000000,0.000000,0.00,0.00,0.00,0.012153,0.00425


## 10. Use the output with GrEAt

In your GrEAt notebook, use:

```python
DATASET_NAME = "csv"
DATASET_PATH = "synthea_patient_level.csv"
target_col = "label"
```

If your loader does not automatically remove ID columns, remove `patient_id` before training or set `KEEP_PATIENT_ID = False` and rerun this notebook.
